# Multi-criteria ambulance routing in Sidi Bel Abbès — data extraction and algorithm benchmark (Track B, v3)

**What this notebook does**
1. Downloads from OpenStreetMap everything relevant to ambulance routing in Sidi Bel Abbès: drivable roads, one-way streets, speed limits, surface, speed bumps (`traffic_calming`), traffic signals, stop and give-way signs, pedestrian crossings, barriers and closed roads. Every OSM bump found is treated as real.
2. Builds per-arc criteria (time, bumps, signals, stop/give-way, crossings, unpaved metres, contraflow metres) and several objective sets (2 to 5 objectives).
3. Runs **published exact multi-objective shortest-path algorithms** on the same queries (patient → nearest emergency hospital: CHU Dr Hassani Abdelkader or EPH Sidi Bel Abbès):
   * official C code of **BOA\*** and **NAMOA\*dr** (bi-objective) — Hernández et al., ICAPS 2020 / AIJ 2023 (github.com/jorgebaier/BOAstar, MIT licence);
   * official C++ code of **EMOA\*** — Ren et al., AIJ 2025 (github.com/rap-lab-org/public_emoa);
   * Python re-implementations from the published pseudocode of **NAMOA\*dr** (Pulido, Mandow, Pérez de la Cruz 2015), **LTMOA\*** (Hernández et al., IJCAI 2023), **BOA\*** and the **Martins** label-setting algorithm (1984), plus the **weighted-sum** scalarisation as a non-exact baseline.
4. Checks that all exact algorithms return the same Pareto set, then records runtime, expansions, solved/timeout and front size for every query.
5. Repeats the benchmark with **dynamic road closures** and with an optional **emergency contraflow** scenario.
6. Exports tables (CSV + LaTeX), figures (PNG + PDF), raw per-query logs and `run_metadata.json` (all assumptions, OSM counts, code commits) in one zip.

**Runtime type:** *Processeur* (CPU). No GPU is used. Expected time: 30–90 min depending on `N_QUERIES` and timeouts.

**Fair-comparison rule:** C/C++ codes are compared with each other; Python codes are compared with each other. Times are never compared across languages.

## 0. Configuration

In [ ]:
# ---- area and hospitals ---------------------------------------------------
CENTER = (35.1919, -0.6298); RADIUS_M = 6000          # same area as runs v1/v2
HOSPITAL_NAME_REGEX = r"\bCHU\b|\bEPH\b"               # confirmed by the authors: CHU + EPH
MANUAL_HOSPITALS = []                                  # optional [("name", lat, lon)]

# ---- speeds and time --------------------------------------------------------
FALLBACK_KMH = 40                                      # FEDZ rule: OSM maxspeed else 40 km/h
BUMP_CROSSING_KMH = 15.0; DECEL_MS2 = 1.5              # ASSUMPTIONS for the slow-down at a bump
TIME_UNIT_S = 0.1                                      # integer costs: time stored in deciseconds

# ---- features -----------------------------------------------------------------
VERTICAL_TYPES = {"yes","bump","hump","table","cushion","speed_bump","speed_hump","dip","rumble_strip","mini_bumps","speed_table"}
UNPAVED = {"unpaved","dirt","earth","ground","gravel","fine_gravel","sand","mud","grass","compacted","pebblestone","rock"}
CLOSED_BARRIERS = {"bollard","block","jersey_barrier","chain","bar","barrier_board","log","planter","debris","rope"}  # impassable for a car
CONTRA_HIGHWAYS = {"residential","living_street","unclassified","tertiary","service"}   # minor one-way streets
RUN_CONTRAFLOW = True                                  # scenario: ambulance may drive against one-way on minor streets (ASSUMPTION, see notes)

# ---- queries and algorithms -------------------------------------------------
SEED = 2026
N_QUERIES = 200                # patient origins (C/C++ codes)
N_QUERIES_PY = 100             # subset used for the Python implementations
TIMEOUT_C_S = 60               # per query
TIMEOUT_PY_S = 60
OBJECTIVE_SETS = {             # name -> list of criteria (first one is always time)
    "T-B":       ["time","bumps"],
    "T-I":       ["time","interruptions"],              # interruptions = signals + stop/give-way
    "T-B-I":     ["time","bumps","interruptions"],
    "T-B-S-St":  ["time","bumps","signals","stops"],
    "T-B-S-St-U":["time","bumps","signals","stops","unpaved_m"],
}
CLOSURE_FRACTIONS = [0.01, 0.03]   # dynamic closures: share of two-way street segments closed at random
WS_LAMBDAS = 25                    # weighted-sum baseline: number of random weight vectors per query
OUT = "fedz_trackB_v3_results"

## 1. Setup (installs osmnx, clones and compiles the official codes)

In [ ]:
import os, sys, json, math, time, random, heapq, itertools, re, subprocess, datetime, zipfile, shutil, warnings
IN_COLAB = "google.colab" in sys.modules
OFFLINE_TEST = os.environ.get("FEDZ_OFFLINE_TEST") == "1"       # only to test this notebook without internet
if IN_COLAB:
    !pip -q install "osmnx>=2.0"
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt
import osmnx as ox, geopandas as gpd
from shapely.geometry import Point
warnings.filterwarnings("ignore", category=FutureWarning); warnings.filterwarnings("ignore", category=DeprecationWarning)
for d in ["", "/fig", "/tab", "/raw", "/graphs"]: os.makedirs(OUT + d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
META = {"run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), "osmnx": ox.__version__,
        "python": sys.version.split()[0], "cpu": (subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip() or "unknown")}
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:])
    return r
BIN = os.path.abspath("bin_src"); os.makedirs(BIN, exist_ok=True)
REPOS = {"BOAstar": "https://github.com/jorgebaier/BOAstar", "public_emoa": "https://github.com/rap-lab-org/public_emoa"}
for name, url in REPOS.items():
    if not os.path.exists(f"{BIN}/{name}"): sh(f"git clone -q {url} {BIN}/{name}")
    META[f"commit_{name}"] = sh(f"git -C {BIN}/{name} rev-parse HEAD").stdout.strip()
# -fno-stack-clash-protection: the official BOA* code declares an 8 MB array on the stack; with the default
# stack-clash probing of recent gcc this adds a constant ~4 ms per run that has nothing to do with the algorithm.
C_FLAGS = "-Wall -O3 -fno-stack-clash-protection"; META["c_flags_BOAstar"] = C_FLAGS
sh(f"mkdir -p {BIN}/BOAstar/obj && make -s -C {BIN}/BOAstar clean && make -s -C {BIN}/BOAstar CFLAGS='{C_FLAGS}' LFLAGS='-Wall -O3'")
sh(f"mkdir -p {BIN}/public_emoa/build && cd {BIN}/public_emoa/build && cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null && make -s -j4 run_emoa")
EXE = {"BOA*": f"{BIN}/BOAstar/boa", "NAMOA*dr": f"{BIN}/BOAstar/namoadr",
       "EMOA*": f"{BIN}/public_emoa/build/run_emoa"}
for k, v in EXE.items(): print(k, "OK" if os.path.exists(v) else "MISSING", v)
savefig = lambda fig, n: (fig.savefig(f"{OUT}/fig/{n}.png", bbox_inches="tight"), fig.savefig(f"{OUT}/fig/{n}.pdf", bbox_inches="tight"))
def savetab(df, n, cap=""):
    df.to_csv(f"{OUT}/tab/{n}.csv")
    try: open(f"{OUT}/tab/{n}.tex", "w").write(df.to_latex(float_format="%.2f", caption=cap, label=f"tab:{n}"))
    except Exception as e: print("latex skipped", e)
print(META)

## 2. OpenStreetMap extraction

In [ ]:
OVERPASS_URLS = ["https://overpass-api.de/api", "https://overpass.private.coffee/api", "https://overpass.kumi.systems/api"]
def robust(fn, *a, **k):
    last = None
    for url in OVERPASS_URLS:
        ox.settings.overpass_url = url
        for i in range(3):
            try: return fn(*a, **k)
            except Exception as e:
                if "No matching features" in str(e) or "InsufficientResponse" in type(e).__name__: raise
                last = e; w = 30 * 2 ** i; print(f"  {url}: {str(e).splitlines()[0][:80]} -> retry in {w}s"); time.sleep(w)
    raise RuntimeError(f"All Overpass servers failed; re-run later. Last error: {last}")
def feats(tags):
    try: return robust(ox.features_from_point, CENTER, tags, dist=RADIUS_M)
    except Exception as e:
        if "No matching features" in str(e) or "InsufficientResponse" in type(e).__name__:
            return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        raise

ox.settings.use_cache = True; ox.settings.log_console = False
ox.settings.useful_tags_node = sorted(set(ox.settings.useful_tags_node) | {"traffic_calming", "barrier", "direction", "traffic_signals:direction", "crossing"})
ox.settings.useful_tags_way = sorted(set(ox.settings.useful_tags_way) | {"surface", "traffic_calming", "motor_vehicle", "vehicle", "access", "oneway", "smoothness"})
if not OFFLINE_TEST:
    Graw = robust(ox.graph_from_point, CENTER, dist=RADIUS_M, network_type="drive", simplify=False)
    # keep every node carrying a tag we need (signals, stops, bumps, barriers, crossings) when simplifying
    G0 = ox.simplify_graph(Graw, node_attrs_include=["highway", "traffic_calming", "barrier"])
    hosp = feats({"amenity": "hospital"}); tc_ways = feats({"traffic_calming": True})
    META["area"] = f"{RADIUS_M} m around {CENTER}"
else:
    from fedz_offline_fixture_v3 import make_fixture
    G0, hosp, tc_ways = make_fixture(); META["area"] = "OFFLINE TEST FIXTURE"
print("raw drive graph:", G0.number_of_nodes(), "nodes", G0.number_of_edges(), "arcs")

In [ ]:
def val(d, k):
    x = d.get(k); return x[0] if isinstance(x, list) else x
def parse_speed(v):
    if isinstance(v, list): v = v[0]
    if v is None or (isinstance(v, float) and math.isnan(v)): return None
    m = re.search(r"(\d+(\.\d+)?)", str(v));
    if not m: return None
    s = float(m.group(1)); return s * 1.609344 if "mph" in str(v) else s

INV = {}
# ---- static closures: impassable barriers and access restrictions -----------
bar = {n: d.get("barrier") for n, d in G0.nodes(data=True) if d.get("barrier")}
INV["barrier_nodes_all"] = len(bar); INV["barrier_types"] = pd.Series(list(bar.values())).value_counts().to_dict()
closed_nodes = [n for n, b in bar.items() if b in CLOSED_BARRIERS]
G1 = G0.copy(); G1.remove_nodes_from(closed_nodes); INV["closed_barrier_nodes_removed"] = len(closed_nodes)
acc_closed = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True)
              if str(val(d, "access")) in {"no", "private"} or str(val(d, "motor_vehicle")) == "no" or str(val(d, "vehicle")) == "no"]
G1.remove_edges_from(acc_closed); INV["access_closed_arcs_removed"] = len(acc_closed)
G1 = ox.truncate.largest_component(G1, strongly=True)

# ---- hospitals -----------------------------------------------------------------
if MANUAL_HOSPITALS:
    H = gpd.GeoDataFrame({"name": [h[0] for h in MANUAL_HOSPITALS]}, geometry=[Point(h[2], h[1]) for h in MANUAL_HOSPITALS], crs="EPSG:4326")
else:
    H = hosp.copy(); H["geometry"] = H.geometry.apply(lambda g: g if g.geom_type == "Point" else g.representative_point())
    H["name"] = H.get("name", pd.Series(["(unnamed)"] * len(H), index=H.index)).fillna("(unnamed)")
    H = H[H.name.str.contains(HOSPITAL_NAME_REGEX, regex=True)].reset_index(drop=True)
assert len(H), "no hospital"; H["node"] = ox.distance.nearest_nodes(G1, H.geometry.x.values, H.geometry.y.values)
META["hospitals"] = H[["name"]].assign(node=H.node.astype(str)).to_dict("records"); print(H[["name", "node"]])

# ---- bumps mapped on ways (not as nodes): mark the matching arcs -------------
way_bumps = set()
if len(tc_ways):
    w = tc_ways[tc_ways.geometry.geom_type != "Point"]
    w = w[w.get("traffic_calming", pd.Series(dtype=str)).astype(str).isin(VERTICAL_TYPES)] if len(w) else w
    if len(w):
        Gp = ox.project_graph(G1); P = w.to_crs(Gp.graph["crs"]).geometry.representative_point()
        E, dist = ox.distance.nearest_edges(Gp, P.x.values, P.y.values, return_dist=True)
        way_bumps = {e for e, d in zip(E, dist) if d <= 15}
INV["bump_ways_matched"] = len(way_bumps)

In [ ]:
# ---- per-arc criteria ------------------------------------------------------------
def node_flags(n):
    d = G1.nodes[n]; hw = str(d.get("highway", "")); tc = str(d.get("traffic_calming", ""))
    return {"bump": int(tc in VERTICAL_TYPES), "signal": int(hw == "traffic_signals"),
            "stop": int(hw in ("stop", "give_way")), "crossing": int(hw == "crossing")}
NF = {n: node_flags(n) for n in G1.nodes}
for k in ["bump", "signal", "stop", "crossing"]: INV[f"nodes_{k}"] = int(sum(f[k] for f in NF.values()))

def bump_delay_s(v_kmh):
    v, vb = v_kmh / 3.6, BUMP_CROSSING_KMH / 3.6
    return 0.0 if v <= vb else (v - vb) ** 2 / (DECEL_MS2 * v)

rows = []
for u, v, k, d in G1.edges(keys=True, data=True):
    spd = parse_speed(d.get("maxspeed")) or FALLBACK_KMH
    f = NF[v]; nb = f["bump"] + int((u, v, k) in way_bumps or (v, u, 0) in way_bumps)
    surf = str(val(d, "surface") or "")
    rows.append(dict(u=u, v=v, k=k, length=d["length"], speed=spd, highway=str(val(d, "highway")), oneway=bool(val(d, "oneway")),
                     surface=surf, bumps=nb, signals=f["signal"], stops=f["stop"], crossings=f["crossing"],
                     unpaved_m=int(round(d["length"])) if surf in UNPAVED else 0, contra_m=0))
ARCS = pd.DataFrame(rows)
ARCS["time_s"] = ARCS.length / (ARCS.speed / 3.6) + ARCS.bumps * ARCS.speed.apply(bump_delay_s)
ARCS["time"] = np.maximum(1, np.round(ARCS.time_s / TIME_UNIT_S)).astype(int)
ARCS["interruptions"] = ARCS.signals + ARCS.stops

INV.update(nodes=G1.number_of_nodes(), arcs=len(ARCS), km_directed=float(ARCS.length.sum() / 1000),
           oneway_arcs_pct=float(100 * ARCS.oneway.mean()), maxspeed_tagged_pct=float(100 * np.mean([d.get("maxspeed") is not None for *_, d in G1.edges(keys=True, data=True)])),
           surface_tagged_pct=float(100 * (ARCS.surface != "").mean()), unpaved_km=float(ARCS.unpaved_m.sum() / 1000),
           arcs_with_bump=int((ARCS.bumps > 0).sum()), arcs_with_signal=int((ARCS.signals > 0).sum()), arcs_with_stop=int((ARCS.stops > 0).sum()))
INVT = pd.Series(INV, dtype=object).to_frame("value"); savetab(INVT, "T1_inventory", "OSM data inventory"); META["inventory"] = INV; INVT

In [ ]:
# ---- contraflow scenario arcs (reverse direction of minor one-way streets) -----
def contra_arcs(A):
    have = set(zip(A.u, A.v)); add = []
    for r in A[A.oneway & A.highway.isin(CONTRA_HIGHWAYS)].itertuples():
        if (r.v, r.u) in have: continue
        f = NF[r.u]
        add.append(dict(u=r.v, v=r.u, k=0, length=r.length, speed=r.speed, highway=r.highway, oneway=False, surface=r.surface,
                        bumps=f["bump"], signals=f["signal"], stops=f["stop"], crossings=f["crossing"], unpaved_m=r.unpaved_m,
                        contra_m=int(round(r.length)), time_s=r.length / (r.speed / 3.6) + f["bump"] * bump_delay_s(r.speed)))
    B = pd.DataFrame(add)
    if len(B): B["time"] = np.maximum(1, np.round(B.time_s / TIME_UNIT_S)).astype(int); B["interruptions"] = B.signals + B.stops
    return B
CONTRA = contra_arcs(ARCS); META["contraflow_arcs_added"] = len(CONTRA); print("contraflow arcs:", len(CONTRA))

# ---- figure: where the criteria are -------------------------------------------
fig, ax = ox.plot_graph(G1, node_size=0, edge_color="#d6d6d6", edge_linewidth=.4, bgcolor="w", show=False, close=False, figsize=(7, 7))
for key, c, m, lab in [("signal", "#1f5aa6", "o", "traffic signals"), ("stop", "#c0392b", "s", "stop / give way"), ("bump", "#000000", "^", "speed bumps")]:
    pts = [(G1.nodes[n]["x"], G1.nodes[n]["y"]) for n, f in NF.items() if f[key]]
    if pts: ax.scatter(*zip(*pts), s=14, c=c, marker=m, label=f"{lab} ({len(pts)})", zorder=5)
ax.scatter(*zip(*[(G1.nodes[n]["x"], G1.nodes[n]["y"]) for n in H.node]), s=90, c="#8e44ad", marker="P", label="CHU / EPH", zorder=6)
ax.legend(loc="lower left", fontsize=7); savefig(fig, "F1_criteria_map"); plt.show()

## 3. Graph export and queries

All algorithms read exactly the same graph: node ids 1..n, integer costs, and parallel arcs split with a dummy node (EMOA\* does not accept parallel arcs).

In [ ]:
class Inst:
    # one instance = arcs table + chosen criteria -> integer cost matrix, adjacency for Python, files for C/C++
    def __init__(self, A, crit, tag):
        self.crit, self.tag = crit, tag
        nodes = sorted(set(A.u) | set(A.v)); self.id = {n: i for i, n in enumerate(nodes)}; n = len(nodes)
        C = A[crit].to_numpy(dtype=np.int64); U = A.u.map(self.id).to_numpy(); V = A.v.map(self.id).to_numpy()
        seen = {}; tails, heads, costs = [], [], []
        for u, v, c in zip(U, V, C):
            if (u, v) in seen:                       # parallel arc -> u -> dummy -> v
                dmy = n; n += 1
                tails += [u, dmy]; heads += [dmy, v]; costs += [c, np.zeros_like(c)]
            else:
                seen[(u, v)] = 1; tails.append(u); heads.append(v); costs.append(c)
        self.n, self.T, self.Hh, self.C = n, np.array(tails), np.array(heads), np.array(costs, dtype=np.int64)
        self.out = [[] for _ in range(n)]; self.inn = [[] for _ in range(n)]
        for e, (u, v) in enumerate(zip(self.T, self.Hh)):
            c = tuple(int(x) for x in self.C[e]); self.out[u].append((v, c)); self.inn[v].append((u, c))
        self.d = len(crit); self.files = None
    def write(self):
        if self.files: return self.files
        base = f"{OUT}/graphs/{self.tag}"; m = len(self.T); fs = []
        for j in range(self.d):
            with open(f"{base}_c{j+1}.gr", "w") as f:
                f.write(f"p sp {self.n} {m}\n"); f.writelines(f"a {u+1} {v+1} {c}\n" for u, v, c in zip(self.T, self.Hh, self.C[:, j]))
            fs.append(f"{base}_c{j+1}.gr")
        if self.d == 2:
            with open(f"{base}_boa.txt", "w") as f:
                f.write(f"{self.n} {m}\n"); f.writelines(f"{u+1} {v+1} {a} {b}\n" for u, v, (a, b) in zip(self.T, self.Hh, self.C))
            fs.append(f"{base}_boa.txt")
        self.files = fs; return fs

def rev_dijkstra(I, t, j):
    d = [math.inf] * I.n; d[t] = 0; pq = [(0, t)]
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]: continue
        for v, c in I.inn[u]:
            nd = du + c[j]
            if nd < d[v]: d[v] = nd; heapq.heappush(pq, (nd, v))
    return d

BASE_T = Inst(ARCS, ["time"], "time_only")
HOSP = sorted({BASE_T.id[h] for h in H.node})
DT = {h: rev_dijkstra(BASE_T, h, 0) for h in HOSP}
rng = random.Random(SEED); cand = [i for i in range(len(BASE_T.id)) if i not in HOSP]
ORIG = rng.sample(cand, min(N_QUERIES, len(cand)))
inv_id = {i: n for n, i in BASE_T.id.items()}
QUERIES = [(inv_id[s], inv_id[min(HOSP, key=lambda h: DT[h][s])]) for s in ORIG if min(DT[h][s] for h in HOSP) < math.inf]
QUERIES_PY = QUERIES[:N_QUERIES_PY]
print(len(QUERIES), "queries (OSM node ids, origin -> nearest hospital)")

## 4. Algorithms

Python implementations follow the published pseudocode; all use exact single-objective reverse-Dijkstra heuristics (consistent), weak dominance (cost-unique Pareto set, as BOA\* and EMOA\*), and include the heuristic computation in the measured time (as the official BOA\* code does).

In [ ]:
class Timeout(Exception): pass
def heur(I, t): return [rev_dijkstra(I, t, j) for j in range(I.d)]
def wdom(a, b):          # a weakly dominates b
    return all(x <= y for x, y in zip(a, b))

def boa_py(I, s, t, deadline):
    h = heur(I, t); h1, h2 = h
    if h1[s] == math.inf: return [], 0
    g2min = [math.inf] * I.n; pq = [(h1[s], h2[s], 0, 0, s)]; sols = []; exp = 0
    while pq:
        f1, f2, g1, g2, u = heapq.heappop(pq)
        if g2 >= g2min[u] or f2 >= g2min[t]: continue
        g2min[u] = g2; exp += 1
        if exp % 2000 == 0 and time.perf_counter() > deadline: raise Timeout
        if u == t: sols.append((g1, g2)); continue
        for v, (a, b) in I.out[u]:
            n1, n2 = g1 + a, g2 + b
            if h1[v] == math.inf or n2 >= g2min[v] or n2 + h2[v] >= g2min[t]: continue
            heapq.heappush(pq, (n1 + h1[v], n2 + h2[v], n1, n2, v))
    return sols, exp

def _mo_astar(I, s, t, deadline, eager):
    # eager=True -> NAMOA*dr (Pulido et al. 2015): open labels at a node dominated by a new label are discarded.
    # eager=False -> LTMOA* (Hernandez et al. 2023): lazy, dominance checked only against closed truncated vectors.
    h = heur(I, t); d = I.d
    if h[0][s] == math.inf: return [], 0
    Gcl = [[] for _ in range(I.n)]; Sol = []; sols = []; exp = 0; tie = itertools.count()
    Gop = [[] for _ in range(I.n)] if eager else None
    g0 = (0,) * d; lab = [g0, s, True]; pq = [(tuple(h[j][s] for j in range(d)), next(tie), lab)]
    if eager: Gop[s].append(lab)
    while pq:
        f, _, lab = heapq.heappop(pq); g, u, alive = lab
        if not alive: continue
        if eager: Gop[u].remove(lab)
        gt, ft = g[1:], f[1:]
        if any(wdom(x, gt) for x in Gcl[u]) or any(wdom(x, ft) for x in Sol): continue
        Gcl[u] = [x for x in Gcl[u] if not wdom(gt, x)] + [gt]; exp += 1
        if exp % 2000 == 0 and time.perf_counter() > deadline: raise Timeout
        if u == t: sols.append(g); Sol = [x for x in Sol if not wdom(gt, x)] + [gt]; continue
        for v, c in I.out[u]:
            if h[0][v] == math.inf: continue
            ng = tuple(a + b for a, b in zip(g, c)); ngt = ng[1:]
            nf = tuple(ng[j] + h[j][v] for j in range(d)); nft = nf[1:]
            if any(wdom(x, ngt) for x in Gcl[v]) or any(wdom(x, nft) for x in Sol): continue
            if eager:
                if any(wdom(o[0], ng) for o in Gop[v]): continue
                for o in Gop[v]:
                    if wdom(ng, o[0]): o[2] = False
                Gop[v] = [o for o in Gop[v] if o[2]]
            nl = [ng, v, True]
            if eager: Gop[v].append(nl)
            heapq.heappush(pq, (nf, next(tie), nl))
    return sols, exp
namoa_dr_py = lambda I, s, t, dl: _mo_astar(I, s, t, dl, True)
ltmoa_py = lambda I, s, t, dl: _mo_astar(I, s, t, dl, False)

def martins_py(I, s, t, deadline):
    # label-setting without heuristics (Martins 1984), lexicographic queue; target pruning only
    perm = [[] for _ in range(I.n)]; pq = [((0,) * I.d, s)]; exp = 0
    while pq:
        g, u = heapq.heappop(pq)
        if any(wdom(x, g) for x in perm[u]) or any(wdom(x, g) for x in perm[t]): continue
        perm[u].append(g); exp += 1
        if exp % 2000 == 0 and time.perf_counter() > deadline: raise Timeout
        if u == t: continue
        for v, c in I.out[u]:
            ng = tuple(a + b for a, b in zip(g, c))
            if not any(wdom(x, ng) for x in perm[v]) and not any(wdom(x, ng) for x in perm[t]): heapq.heappush(pq, (ng, v))
    return perm[t], exp

def weighted_sum_py(I, s, t, deadline, W=None):
    # non-exact baseline: Dijkstra on random weighted sums of normalised criteria; returns distinct points found
    rngw = np.random.default_rng(s); scale = np.maximum(1, I.C.max(0)).astype(float); found = set(); exp = 0
    for _ in range(WS_LAMBDAS):
        w = rngw.dirichlet(np.ones(I.d)) / scale
        dist = {s: 0.0}; vec = {s: (0,) * I.d}; pq = [(0.0, s)]; done = set()
        while pq:
            du, u = heapq.heappop(pq)
            if u in done: continue
            done.add(u); exp += 1
            if u == t: break
            for v, c in I.out[u]:
                nd = du + float(np.dot(w, c))
                if nd < dist.get(v, math.inf): dist[v] = nd; vec[v] = tuple(a + b for a, b in zip(vec[u], c)); heapq.heappush(pq, (nd, v))
        if t in vec: found.add(vec[t])
        if time.perf_counter() > deadline: raise Timeout
    nd_ = [p for p in found if not any(q != p and wdom(q, p) for q in found)]
    return nd_, exp

PY_ALGOS = {"BOA* (py)": (boa_py, 2), "NAMOA*dr (py)": (namoa_dr_py, None), "LTMOA* (py)": (ltmoa_py, None),
            "Martins (py)": (martins_py, None), "Weighted sum (py, non-exact)": (weighted_sum_py, None)}

def run_c(alg, I, s, t):
    fs = I.write(); S, T_ = s + 1, t + 1
    try:
        if alg in ("BOA*", "NAMOA*dr"):
            r = subprocess.run([EXE[alg], fs[-1], str(S), str(T_)], capture_output=True, text=True, timeout=TIMEOUT_C_S)
            x = r.stdout.strip().splitlines()[-1].split(";")
            return dict(n_sol=int(x[2]), ms=float(x[3]), generated=int(x[4]), expanded=int(x[5]), status="ok")
        if alg == "EMOA*":
            res = f"{OUT}/graphs/_emoa_res.txt"
            subprocess.run([EXE[alg], str(S), str(T_), str(TIMEOUT_C_S), str(I.d), *fs[:I.d], res], capture_output=True, text=True, timeout=TIMEOUT_C_S + 30)
            kv = {}; costs = []
            lines = open(res).read().splitlines()
            for i, ln in enumerate(lines):
                if ":" in ln and not ln.startswith("Label"): a, b = ln.split(":", 1); kv[a.strip()] = b.strip()
                if ln.startswith("Label"): costs.append(tuple(int(float(z)) for z in re.findall(r"-?\d+\.?\d*", lines[i + 1])))
            to = kv.get("timeout", "0") not in ("0", "false")
            return dict(n_sol=int(kv["N"]), ms=1000 * (float(kv["rt_initHeu"]) + float(kv["rt_search"])), generated=int(kv["n_generated"]),
                        expanded=int(kv["n_expanded"]), status="timeout" if to else "ok", front=sorted(costs))
    except subprocess.TimeoutExpired:
        return dict(status="timeout")
    except Exception as e:
        return dict(status=f"error: {e}")

def run_py(alg, I, s, t):
    fn, dreq = PY_ALGOS[alg]
    if dreq and I.d != dreq: return None
    t0 = time.perf_counter()
    try:
        sols, exp = fn(I, s, t, t0 + TIMEOUT_PY_S); return dict(n_sol=len(sols), ms=1000 * (time.perf_counter() - t0), expanded=exp, status="ok", front=sorted(tuple(map(int, x)) for x in sols))
    except Timeout:
        return dict(status="timeout", ms=1000 * TIMEOUT_PY_S)

## 5. Benchmark on the OSM network

In [ ]:
def benchmark(A, scenario, sets=OBJECTIVE_SETS):
    recs = []
    for sname, crit in sets.items():
        I = Inst(A, crit, f"{scenario}_{sname}".replace(" ", "_").replace("%", "pct"))
        qs = [(I.id[a], I.id[b]) for a, b in QUERIES if a in I.id and b in I.id]
        qpy = set(qs[:N_QUERIES_PY]); t0 = time.time()
        for qi, (s, t) in enumerate(qs):
            row_front = {}; row_n = {}
            for alg in (["BOA*", "NAMOA*dr"] if I.d == 2 else []) + ["EMOA*"]:
                r = run_c(alg, I, s, t)
                if r.get("status") == "ok": row_n[alg] = r["n_sol"]
                recs.append(dict(scenario=scenario, set=sname, d=I.d, q=qi, alg=alg, lang="C/C++", **{k: v for k, v in r.items() if k != "front"}))
                if "front" in r: row_front[alg] = r["front"]
            if (s, t) in qpy:
                for alg in PY_ALGOS:
                    r = run_py(alg, I, s, t)
                    if r is None: continue
                    recs.append(dict(scenario=scenario, set=sname, d=I.d, q=qi, alg=alg, lang="Python", **{k: v for k, v in r.items() if k != "front"}))
                    if "front" in r: row_front[alg] = r["front"]
                    if r.get("status") == "ok" and "non-exact" not in alg: row_n[alg] = r["n_sol"]
            if len(set(row_n.values())) > 1:
                recs.append(dict(scenario=scenario, set=sname, d=I.d, q=qi, alg=str(row_n), lang="CHECK", status="MISMATCH"))
            # exactness check: every exact algorithm that finished must return the same cost set
            exact = {k: v for k, v in row_front.items() if "non-exact" not in k}
            if len(exact) > 1:
                ref = next(iter(exact.values()))
                for k, v in exact.items():
                    if v != ref: recs.append(dict(scenario=scenario, set=sname, d=I.d, q=qi, alg=k, lang="CHECK", status="MISMATCH"))
            if "Weighted sum (py, non-exact)" in row_front and exact:
                ref = set(next(iter(exact.values()))); ws = set(row_front["Weighted sum (py, non-exact)"])
                recs.append(dict(scenario=scenario, set=sname, d=I.d, q=qi, alg="WS coverage", lang="CHECK", status="ok",
                                 coverage=len(ref & ws) / len(ref) if ref else 1.0, front_exact=len(ref)))
        print(f"{scenario:>14} | {sname:<11} d={I.d} | {len(qs)} queries | {time.time()-t0:.0f}s", flush=True)
    return recs

R = benchmark(ARCS, "OSM")
RAW = pd.DataFrame(R); RAW.to_csv(f"{OUT}/raw/bench_OSM.csv", index=False)
print("mismatches:", int((RAW.status == "MISMATCH").sum()))

In [ ]:
def perf_table(D):
    D = D[D.lang != "CHECK"]
    g = D.groupby(["scenario", "set", "d", "lang", "alg"], sort=False)
    T = g.apply(lambda x: pd.Series({
        "queries": len(x), "solved_%": 100 * (x.status == "ok").mean(),
        "ms_median": x.loc[x.status == "ok", "ms"].median(), "ms_mean": x.loc[x.status == "ok", "ms"].mean(),
        "ms_p95": x.loc[x.status == "ok", "ms"].quantile(.95), "ms_max": x.loc[x.status == "ok", "ms"].max(),
        "expanded_median": x.loc[x.status == "ok", "expanded"].median() if "expanded" in x else np.nan,
        "front_mean": x.loc[x.status == "ok", "n_sol"].mean(), "front_max": x.loc[x.status == "ok", "n_sol"].max()}), include_groups=False)
    return T
T2 = perf_table(RAW); savetab(T2, "T2_performance_OSM", "Algorithm performance on the Sidi Bel Abbes OSM network"); T2.round(2)

In [ ]:
# winner per objective set (exact algorithms, all queries solved, lowest median time) -- per language
def winners(T):
    out = []
    for (sc, st, d, lang), x in T.reset_index().groupby(["scenario", "set", "d", "lang"], sort=False):
        x = x[~x.alg.str.contains("non-exact")]
        full = x[x["solved_%"] == 100]
        pick = (full if len(full) else x).sort_values(["solved_%", "ms_median"], ascending=[False, True]).iloc[0]
        out.append(dict(scenario=sc, set=st, d=d, lang=lang, best=pick.alg, ms_median=pick.ms_median, solved_pct=pick["solved_%"],
                        runner_up=(full.sort_values("ms_median").iloc[1].alg if len(full) > 1 else "-")))
    return pd.DataFrame(out).set_index(["scenario", "set", "lang"])
T3 = winners(T2); savetab(T3, "T3_best_algorithm_OSM", "Fastest exact algorithm per objective set"); T3

# weighted-sum coverage of the exact Pareto set
C = RAW[RAW.alg == "WS coverage"].groupby(["scenario", "set", "d"]).agg(queries=("q", "size"), coverage_mean_pct=("coverage", lambda x: 100 * x.mean()),
                                                                     share_full_coverage_pct=("coverage", lambda x: 100 * (x >= 1).mean()), front_mean=("front_exact", "mean"))
savetab(C, "T4_weighted_sum_coverage", "Share of the exact Pareto set found by weighted sums"); display(C.round(1))

# figure: runtime vs number of objectives
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4), sharey=False)
for ax, lang in zip(axs, ["C/C++", "Python"]):
    x = T2.reset_index(); x = x[(x.lang == lang) & (~x.alg.str.contains("non-exact"))]
    for alg, y in x.groupby("alg"):
        y = y.groupby("d").ms_median.median(); ax.plot(y.index, y.values, marker="o", label=alg)
    ax.set_yscale("log"); ax.set_xlabel("number of objectives"); ax.set_ylabel("median runtime per query (ms)"); ax.set_title(lang, fontsize=9); ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); savefig(fig, "F2_runtime_vs_objectives"); plt.show()

# figure: runtime distribution per algorithm for each objective set (C/C++)
x = RAW[(RAW.lang == "C/C++") & (RAW.status == "ok")]
fig, ax = plt.subplots(figsize=(9, 3.4)); labels = []; data = []
for (st, alg), y in x.groupby(["set", "alg"], sort=False): labels.append(f"{st}\n{alg}"); data.append(y.ms.values)
ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels, fontsize=6, rotation=90)
ax.set_yscale("log"); ax.set_ylabel("runtime per query (ms)"); fig.tight_layout(); savefig(fig, "F3_runtime_boxplots_C"); plt.show()

## 6. Dynamic road closures

A share of street segments (both directions) is closed at random, as when a road is blocked. The same queries are re-solved; queries whose origin is cut off are dropped.

In [ ]:
RC = []; IMPACT = []
base_fast = {}
I0 = Inst(ARCS, ["time"], "tmp_fast")
for a, b in QUERIES: base_fast[(a, b)] = rev_dijkstra(I0, I0.id[b], 0)[I0.id[a]]
for frac in CLOSURE_FRACTIONS:
    r = np.random.default_rng(int(frac * 1e4)); pairs = sorted({tuple(sorted((u, v))) for u, v in zip(ARCS.u, ARCS.v)})
    closed = {pairs[i] for i in r.choice(len(pairs), int(frac * len(pairs)), replace=False)}
    A = ARCS[[tuple(sorted((u, v))) not in closed for u, v in zip(ARCS.u, ARCS.v)]]
    Gc = nx.DiGraph(); Gc.add_edges_from(zip(A.u, A.v)); scc = max(nx.strongly_connected_components(Gc), key=len)
    A = A[A.u.isin(scc) & A.v.isin(scc)]
    tag = f"closed {100*frac:g}%"
    RC += benchmark(A, tag, {k: v for k, v in OBJECTIVE_SETS.items() if k in ("T-B", "T-B-I", "T-B-S-St")})
    Ic = Inst(A, ["time"], "tmp_fast_c")
    for a, b in QUERIES:
        if a in Ic.id and b in Ic.id:
            tc = rev_dijkstra(Ic, Ic.id[b], 0)[Ic.id[a]]
            IMPACT.append(dict(scenario=tag, delay_pct=100 * (tc / base_fast[(a, b)] - 1)))
RCD = pd.DataFrame(RC); RCD.to_csv(f"{OUT}/raw/bench_closures.csv", index=False)
T5 = perf_table(RCD); savetab(T5, "T5_performance_closures", "Performance with dynamic closures"); display(T5.round(2))
T5b = pd.DataFrame(IMPACT).groupby("scenario").delay_pct.describe(); savetab(T5b, "T5b_closure_delay", "Fastest-time increase caused by closures"); T5b.round(2)

## 7. Emergency contraflow scenario (assumption)

Adds the reverse direction of minor one-way streets with an extra criterion *contraflow metres*. Whether and where an ambulance may legally and safely drive against traffic must be confirmed with the local authorities; this is a what-if scenario.

In [ ]:
if RUN_CONTRAFLOW and len(CONTRA):
    AC = pd.concat([ARCS, CONTRA], ignore_index=True)
    RCF = benchmark(AC, "contraflow", {"T-C": ["time", "contra_m"], "T-B-C": ["time", "bumps", "contra_m"], "T-B-I-C": ["time", "bumps", "interruptions", "contra_m"]})
    RCF = pd.DataFrame(RCF); RCF.to_csv(f"{OUT}/raw/bench_contraflow.csv", index=False)
    T6 = perf_table(RCF); savetab(T6, "T6_performance_contraflow", "Performance in the contraflow scenario"); display(T6.round(2))
    Ic = Inst(AC, ["time"], "tmp_fast_cf"); sav = []
    for a, b in QUERIES:
        tc = rev_dijkstra(Ic, Ic.id[b], 0)[Ic.id[a]]; sav.append(100 * (1 - tc / base_fast[(a, b)]))
    T6b = pd.Series(sav).describe().to_frame("fastest_time_saving_%"); savetab(T6b, "T6b_contraflow_saving", "Time saved if contraflow is allowed"); display(T6b.round(2))

## 8. Export

In [ ]:
ALL = pd.concat([RAW, RCD] + ([RCF] if RUN_CONTRAFLOW and len(CONTRA) else []), ignore_index=True)
META["mismatches_total"] = int((ALL.status == "MISMATCH").sum())
META["config"] = dict(CENTER=CENTER, RADIUS_M=RADIUS_M, FALLBACK_KMH=FALLBACK_KMH, BUMP_CROSSING_KMH=BUMP_CROSSING_KMH, DECEL_MS2=DECEL_MS2,
                      TIME_UNIT_S=TIME_UNIT_S, VERTICAL_TYPES=sorted(VERTICAL_TYPES), UNPAVED=sorted(UNPAVED), CLOSED_BARRIERS=sorted(CLOSED_BARRIERS),
                      CONTRA_HIGHWAYS=sorted(CONTRA_HIGHWAYS), SEED=SEED, N_QUERIES=N_QUERIES, N_QUERIES_PY=N_QUERIES_PY,
                      TIMEOUT_C_S=TIMEOUT_C_S, TIMEOUT_PY_S=TIMEOUT_PY_S, OBJECTIVE_SETS=OBJECTIVE_SETS, CLOSURE_FRACTIONS=CLOSURE_FRACTIONS, WS_LAMBDAS=WS_LAMBDAS)
json.dump(META, open(f"{OUT}/run_metadata.json", "w"), indent=2, default=str)
ARCS.drop(columns=["k"]).to_csv(f"{OUT}/raw/arcs_criteria.csv", index=False)
if not OFFLINE_TEST: ox.save_graphml(G1, f"{OUT}/sba_drive_network_v3.graphml")
zf = f"{OUT}.zip"
with zipfile.ZipFile(zf, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for f in fs:
            if not (f.endswith(".gr") or f.endswith("_boa.txt") or f.startswith("_emoa")): z.write(os.path.join(root, f))
print("mismatches:", META["mismatches_total"], "| written", zf, round(os.path.getsize(zf) / 1e6, 1), "MB")
if IN_COLAB:
    from google.colab import files; files.download(zf)

## Notes for the paper
* Signals and stop/give-way signs are OSM nodes counted when an arc **enters** that node; the direction of a stop sign is not modelled (stated as a limitation).
* One-way streets are respected by the network; contraflow is only a labelled what-if scenario.
* Static closures: impassable barrier nodes and `access=no/private` arcs are removed; dynamic closures are random and seeded.
* Report C/C++ and Python timings separately. All exact algorithms must show 0 mismatches (`run_metadata.json`).
* Other published algorithms (e.g. BOBA\*, T-MDA, NWMOA\*/LTMOA\* official codes) can be added the same way if their code is obtained.